# Retry các URL lỗi anti-bot bằng Crawl4AI

Notebook này chạy trực tiếp trên **Kaggle**, đọc dữ liệu từ `/kaggle/input` rồi ghi toàn bộ kết quả vào `/kaggle/working/retry-antibot-output`. Input gốc luôn được giữ nguyên vì `/kaggle/input` là read-only.

## Thiết lập bắt buộc trên Kaggle

### 1. Thêm một Kaggle Dataset làm Input

Chỉ dùng **một folder cha duy nhất** và đặt trực tiếp ba file sau trong folder đó (không cần thêm folder con):

```text
/kaggle/input/<ten-dataset>/
├── crawl_attempts.jsonl
├── dataset.jsonl
└── domain_summary.csv
```

Notebook sẽ tự tìm folder duy nhất chứa đủ ba file. Không cần sửa đường dẫn nếu cấu trúc đúng như trên.

Vai trò của từng file:

- `domain_summary.csv`: lấy các domain có cột `successes = 0`.
- `crawl_attempts.jsonl`: lấy toàn bộ URL lỗi của các domain cần retry và metadata gốc.
- `dataset.jsonl`: dataset thành công hiện tại; được giữ nguyên và dùng làm nền cho dataset mới.

### 2. Kaggle Notebook Settings

- **Internet: On** để cài Crawl4AI, Chromium và truy cập website cần crawl.
- **Accelerator: None**; notebook không cần GPU.
- Chạy notebook theo thứ tự từ trên xuống. Cell cài đặt chỉ cần chạy một lần trong mỗi session.

### 3. File đầu ra

Notebook tạo các file trong `/kaggle/working/retry-antibot-output/`:

- `dataset.jsonl`: 791 record thành công ban đầu cộng các URL retry thành công.
- `retry_success.jsonl`: kết quả retry thành công, mỗi URL một record mới nhất.
- `retry_errors.jsonl`: URL vẫn lỗi, mỗi URL một record mới nhất.
- `retry_events.jsonl`: log các sự kiện crawl thành công.

Cuối notebook còn tạo `/kaggle/working/retry-antibot-results.zip` để tải toàn bộ kết quả.

## Nguyên tắc crawl
- Tôn trọng `robots.txt`, điều khoản sử dụng và rate limit của website.
- Không tự giải CAPTCHA và không cố vượt trang yêu cầu đăng nhập/quyền truy cập.
- Mỗi domain chỉ crawl tuần tự, có jitter và exponential backoff.
- Giữ cùng browser profile, cookie và user-agent trong cả phiên retry.
- Kết quả cuối gồm `dataset.jsonl`, các log retry và file ZIP để tải xuống.

## 0. Cài đặt

Kaggle cần bật **Internet**. Chạy cell này một lần ở đầu session để cài Crawl4AI và Chromium.

In [ ]:
%pip install -q -U "crawl4ai>=0.9.4,<0.10" pandas tqdm
!crawl4ai-setup


## 1. Tham số retry

In [ ]:
from pathlib import Path
import asyncio
import hashlib
import json
import random
from datetime import datetime, timezone
from urllib.parse import urlparse

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

IS_KAGGLE = Path("/kaggle/input").exists() and Path("/kaggle/working").exists()

# Để None khi chỉ có một folder input chứa đủ ba file (thiết lập khuyến nghị).
# Chỉ đặt path thủ công nếu cần debug một input cụ thể.
# Ví dụ: Path("/kaggle/input/my-crawl-data")
KAGGLE_INPUT_DIR = None

def find_input_dir():
    if IS_KAGGLE:
        if KAGGLE_INPUT_DIR is not None:
            candidates = [Path(KAGGLE_INPUT_DIR)]
        else:
            candidates = sorted({
                path.parent for path in Path("/kaggle/input").rglob("crawl_attempts.jsonl")
                if (path.parent / "dataset.jsonl").exists()
                and (path.parent / "domain_summary.csv").exists()
            })
        if len(candidates) != 1:
            paths = "\n".join(map(str, candidates)) or "<không tìm thấy>"
            raise RuntimeError(
                "Cần đúng một thư mục chứa crawl_attempts.jsonl, dataset.jsonl và domain_summary.csv. "
                f"Các thư mục tìm thấy:\n{paths}\nHãy đặt KAGGLE_INPUT_DIR thủ công."
            )
        return candidates[0].resolve()

    candidates = (
        Path.cwd(),
        Path.cwd() / "md-urls-pattern-crawl-output",
        Path.cwd() / "domain_cleanliness" / "md-urls-pattern-crawl-output",
    )
    for path in candidates:
        if all((path / name).exists() for name in ("crawl_attempts.jsonl", "dataset.jsonl", "domain_summary.csv")):
            return path.resolve()
    raise FileNotFoundError("Không tìm thấy thư mục chứa crawl_attempts.jsonl, dataset.jsonl và domain_summary.csv")

INPUT_DIR = find_input_dir()
OUTPUT_DIR = Path("/kaggle/working/retry-antibot-output") if IS_KAGGLE else INPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ATTEMPTS_PATH = INPUT_DIR / "crawl_attempts.jsonl"
DOMAIN_SUMMARY_PATH = INPUT_DIR / "domain_summary.csv"
SOURCE_DATASET_PATH = INPUT_DIR / "dataset.jsonl"
DATASET_PATH = OUTPUT_DIR / "dataset.jsonl"
SUCCESS_PATH = OUTPUT_DIR / "retry_success.jsonl"
ERROR_PATH = OUTPUT_DIR / "retry_errors.jsonl"
EVENT_LOG_PATH = OUTPUT_DIR / "retry_events.jsonl"
PROFILE_DIR = OUTPUT_DIR / ".crawl4ai-retry-profile"

# Kaggle input là read-only: tạo working copy để merge và resume trong session.
if DATASET_PATH.resolve() != SOURCE_DATASET_PATH.resolve() and not DATASET_PATH.exists():
    import shutil
    shutil.copy2(SOURCE_DATASET_PATH, DATASET_PATH)

DRY_RUN = False                 # True = chỉ preview, False = chạy retry toàn bộ
MAX_URLS = None                  # None = tất cả; nên tăng dần 20 -> 100 -> all
MAX_URLS_PER_DOMAIN = None        # Pilot nhỏ trước khi retry cả domain
OUTER_ATTEMPTS = 2             # Số vòng retry do notebook quản lý
MIN_DELAY_SECONDS = 8.0
MAX_DELAY_SECONDS = 15.0
BLOCK_BACKOFF_SECONDS = (45.0, 90.0)
HEADLESS = True                # False để debug trực quan trên máy local
USE_PERSISTENT_PROFILE = True  # Giữ cookie/localStorage giữa các lần chạy

print({
    "environment": "kaggle" if IS_KAGGLE else "local",
    "input_dir": str(INPUT_DIR),
    "output_dir": str(OUTPUT_DIR),
    "dataset_working_copy": str(DATASET_PATH),
})


## 2. Chọn URL cần retry

Đọc danh sách domain có `successes = 0` trực tiếp từ `domain_summary.csv`, sau đó retry **toàn bộ URL lỗi** của các domain này trong `crawl_attempts.jsonl`. URL đã retry thành công trước đó sẽ tự động được bỏ qua.

In [ ]:
def read_jsonl(path):
    if not path.exists():
        return pd.DataFrame()
    rows = []
    with path.open(encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON lỗi tại {path.name}:{line_no}") from exc
    return pd.DataFrame(rows)

attempts = read_jsonl(ATTEMPTS_PATH)
attempts["success"] = attempts["success"].fillna(False).astype(bool)
domain_col = "sampled_domain" if "sampled_domain" in attempts.columns else "domain"

domain_summary = pd.read_csv(DOMAIN_SUMMARY_PATH, encoding="utf-8-sig")
required_summary_columns = {"domain", "successes"}
missing_summary_columns = required_summary_columns.difference(domain_summary.columns)
if missing_summary_columns:
    raise ValueError(f"domain_summary.csv thiếu cột: {sorted(missing_summary_columns)}")
domain_summary["successes"] = pd.to_numeric(domain_summary["successes"], errors="raise")
zero_domains = set(domain_summary.loc[domain_summary["successes"].eq(0), "domain"].dropna().astype(str))

candidates = attempts.loc[
    (~attempts["success"])
    & attempts[domain_col].astype(str).isin(zero_domains)
    & attempts["url"].notna()
    & attempts["url"].astype(str).str.strip().ne("")
].copy()

previous_success = read_jsonl(SUCCESS_PATH)
done_urls = set(previous_success.get("url", pd.Series(dtype=str)).dropna().astype(str))
candidates["url"] = candidates["url"].astype(str).str.strip()
candidates = candidates.loc[~candidates["url"].isin(done_urls)]
candidates = candidates.drop_duplicates("url").sort_values([domain_col, "url"])
if MAX_URLS_PER_DOMAIN is not None:
    candidates = candidates.groupby(domain_col, group_keys=False).head(MAX_URLS_PER_DOMAIN)
if MAX_URLS is not None:
    candidates = candidates.head(MAX_URLS)

candidate_domains = set(candidates[domain_col].dropna().astype(str))
assert candidate_domains.issubset(zero_domains), "Có URL nằm ngoài nhóm zero-success"
print(f"Domain trong summary: {len(domain_summary)}")
print(f"Zero-success domains: {len(zero_domains)}")
print(f"URL sẽ retry (sau resume/limit): {len(candidates)}")
display(candidates[[c for c in [domain_col, "url", "status_code", "final_status_code", "error_type", "error"] if c in candidates.columns]])


## 3. Cấu hình Crawl4AI

Cấu hình khuyến nghị cho nhóm lỗi này:

- `enable_stealth=True`: giảm các dấu hiệu automation cơ bản.
- Persistent context: tái sử dụng cookie/localStorage; không đổi fingerprint mỗi URL.
- `magic`, `simulate_user`, `override_navigator`: chỉ bật cho batch retry nhỏ.
- `max_retries=2`: Crawl4AI tự nhận diện một số challenge và thử lại có giới hạn.
- `check_robots_txt=True`: không crawl URL bị robots cấm.
- Không dùng `arun_many` cho retry anti-bot: session theo domain cần chạy tuần tự.

In [ ]:
from crawl4ai import AsyncWebCrawler, BrowserConfig, CacheMode, CrawlerRunConfig
try:
    from crawl4ai.content_filter_strategy import PruningContentFilterLXML as PruningFilter
except ImportError:  # Crawl4AI 0.8.x
    from crawl4ai.content_filter_strategy import PruningContentFilter as PruningFilter
from crawl4ai.markdown_generation_strategy import DefaultMarkdownGenerator

browser_kwargs = dict(
    browser_type="chromium",
    headless=HEADLESS,
    enable_stealth=True,
    viewport_width=1365,
    viewport_height=768,
    headers={
        "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
        "DNT": "1",
    },
    verbose=False,
)
if USE_PERSISTENT_PROFILE:
    browser_kwargs.update(
        use_persistent_context=True,
        user_data_dir=str(PROFILE_DIR),
    )
browser_config = BrowserConfig(**browser_kwargs)

prune_filter = PruningFilter(
    threshold=0.6, threshold_type="dynamic", min_word_threshold=25
)
markdown_generator = DefaultMarkdownGenerator(
    content_source="cleaned_html",
    content_filter=prune_filter,
    options={"ignore_links": True, "ignore_images": True},
)

def session_id_for(domain):
    digest = hashlib.sha1(domain.encode("utf-8")).hexdigest()[:12]
    return f"retry-{digest}"

def make_run_config(domain):
    return CrawlerRunConfig(
        cache_mode=CacheMode.BYPASS,
        session_id=session_id_for(domain),
        check_robots_txt=True,
        wait_until="domcontentloaded",
        page_timeout=90_000,
        wait_for="js:() => document.body && document.body.innerText && document.body.innerText.length > 300",
        wait_for_timeout=25_000,
        delay_before_return_html=3.0,
        magic=True,
        simulate_user=True,
        override_navigator=True,
        remove_overlay_elements=True,
        remove_consent_popups=True,
        max_retries=2,
        locale="vi-VN",
        timezone_id="Asia/Ho_Chi_Minh",
        excluded_tags=["nav", "footer", "aside"],
        markdown_generator=markdown_generator,
        verbose=False,
    )


## 4. Phát hiện challenge và chuẩn hoá kết quả

In [ ]:
BLOCK_MARKERS = (
    "checking your browser", "just a moment", "verify you are human",
    "cf-chl-", "captcha", "access denied", "request blocked",
)

def markdown_text(result):
    markdown = getattr(result, "markdown", None)
    if isinstance(markdown, str):
        return markdown
    return (getattr(markdown, "fit_markdown", None) or
            getattr(markdown, "raw_markdown", None) or "")

def challenge_reason(result, markdown):
    status = getattr(result, "status_code", None)
    html = (getattr(result, "html", None) or "").lower()
    text = (markdown or "").lower()
    if status in {403, 429, 503}:
        return f"http_{status}"
    if any(marker in html[:100_000] or marker in text[:10_000] for marker in BLOCK_MARKERS):
        return "anti_bot_challenge"
    if len(text.strip()) < 100:
        return "empty_or_too_short"
    return None

def result_record(result, source_row, outer_attempt):
    requested_url = source_row["url"]
    final_url = getattr(result, "redirected_url", None) or getattr(result, "url", None) or requested_url
    markdown = markdown_text(result)
    reason = challenge_reason(result, markdown)
    metadata = getattr(result, "metadata", None) or {}
    success = bool(getattr(result, "success", False)) and reason is None
    record = {
        "url": requested_url,
        "final_url": final_url,
        "domain": urlparse(final_url).netloc.lower(),
        "sampled_domain": source_row.get(domain_col),
        "domain_id": source_row.get("domain_id"),
        "doc_id": source_row.get("doc_id"),
        "status_code": getattr(result, "status_code", None),
        "success": success,
        "retry_attempt": outer_attempt,
        "retried_at": datetime.now(timezone.utc).isoformat(),
        "title": metadata.get("title"),
    }
    if success:
        record.update(markdown=markdown, markdown_chars=len(markdown), error_type=None, error=None)
    else:
        record.update(
            error_type=reason or "crawl_error",
            error=getattr(result, "error_message", None) or reason or "Unknown crawl error",
            response_preview=(markdown or getattr(result, "html", None) or "")[:1000],
        )
    return record


## 5. Chạy retry

Notebook append từng kết quả ngay sau khi crawl xong để có checkpoint. Khi vẫn gặp CAPTCHA/challenge, nó ghi lỗi và backoff; **không cố giải CAPTCHA**.

In [ ]:
def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
        f.flush()

def log_crawl_success(record):
    event = {
        "event": "crawl_success",
        "logged_at": datetime.now(timezone.utc).isoformat(),
        "url": record.get("url"),
        "final_url": record.get("final_url"),
        "sampled_domain": record.get("sampled_domain"),
        "domain_id": record.get("domain_id"),
        "doc_id": record.get("doc_id"),
        "status_code": record.get("status_code"),
        "retry_attempt": record.get("retry_attempt"),
        "markdown_chars": record.get("markdown_chars", 0),
    }
    append_jsonl(EVENT_LOG_PATH, event)
    tqdm.write(
        "[SUCCESS] "
        f"domain={event['sampled_domain']} "
        f"status={event['status_code']} "
        f"attempt={event['retry_attempt']}/{OUTER_ATTEMPTS} "
        f"chars={event['markdown_chars']:,} "
        f"url={event['url']}"
    )

async def retry_candidates(rows):
    summary = {"success": 0, "failed": 0}
    current_domain = None
    async with AsyncWebCrawler(config=browser_config) as crawler:
        for source_row in tqdm(rows.to_dict("records"), desc="Retry anti-bot"):
            domain = source_row.get(domain_col) or urlparse(source_row["url"]).netloc.lower()
            if current_domain is not None and domain != current_domain:
                await crawler.crawler_strategy.kill_session(session_id_for(current_domain))
            current_domain = domain
            final_record = None

            for outer_attempt in range(1, OUTER_ATTEMPTS + 1):
                try:
                    result = await crawler.arun(
                        url=source_row["url"],
                        config=make_run_config(domain),
                    )
                    final_record = result_record(result, source_row, outer_attempt)
                except Exception as exc:
                    final_record = {
                        "url": source_row["url"],
                        "sampled_domain": domain,
                        "domain_id": source_row.get("domain_id"),
                        "doc_id": source_row.get("doc_id"),
                        "success": False,
                        "retry_attempt": outer_attempt,
                        "retried_at": datetime.now(timezone.utc).isoformat(),
                        "error_type": type(exc).__name__,
                        "error": str(exc),
                    }

                if final_record["success"]:
                    break
                if outer_attempt < OUTER_ATTEMPTS:
                    await asyncio.sleep(random.uniform(*BLOCK_BACKOFF_SECONDS) * (2 ** (outer_attempt - 1)))

            append_jsonl(SUCCESS_PATH if final_record["success"] else ERROR_PATH, final_record)
            summary["success" if final_record["success"] else "failed"] += 1
            if final_record["success"]:
                log_crawl_success(final_record)
            await asyncio.sleep(random.uniform(MIN_DELAY_SECONDS, MAX_DELAY_SECONDS))

        if current_domain is not None:
            await crawler.crawler_strategy.kill_session(session_id_for(current_domain))
    return summary

if DRY_RUN:
    print("DRY_RUN=True: chưa gửi request. Kiểm tra preview ở trên rồi đổi thành False.")
else:
    run_summary = await retry_candidates(candidates)
    print(run_summary)


## 6. Đánh giá kết quả retry

In [ ]:
def latest_records_by_url(frame):
    if frame.empty or "url" not in frame.columns:
        return frame.copy()
    ordered = frame.copy()
    if "retried_at" in ordered.columns:
        ordered = ordered.sort_values("retried_at", kind="stable")
    return ordered.drop_duplicates("url", keep="last")

def rewrite_jsonl_frame(path, frame):
    temp_path = path.with_name(f".{path.name}.tmp")
    with temp_path.open("w", encoding="utf-8") as f:
        for record in frame.to_dict("records"):
            f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
    temp_path.replace(path)

retry_success = read_jsonl(SUCCESS_PATH)
retry_errors = read_jsonl(ERROR_PATH)
retry_success = latest_records_by_url(retry_success)
retry_errors = latest_records_by_url(retry_errors)
recovered_urls = set(retry_success.get("url", pd.Series(dtype=str)).dropna().astype(str))
if not retry_errors.empty:
    retry_errors = retry_errors.loc[~retry_errors["url"].astype(str).isin(recovered_urls)].copy()
rewrite_jsonl_frame(SUCCESS_PATH, retry_success)
rewrite_jsonl_frame(ERROR_PATH, retry_errors)
print({
    "retry_success_rows": len(retry_success),
    "retry_error_rows": len(retry_errors),
    "unique_urls_recovered": len(recovered_urls),
})
if not retry_errors.empty:
    display(retry_errors.groupby(["sampled_domain", "error_type"], dropna=False).size().rename("count").reset_index())


## 7. Merge kết quả thành công vào `dataset.jsonl`

Quy tắc merge theo `url`:

- URL đã có trong dataset: **thay thế** nội dung crawl bằng kết quả retry mới nhất, đồng thời giữ metadata cũ nếu retry không trả lại trường đó.
- URL chưa có: **append** vào cuối dataset, bổ sung metadata từ `crawl_attempts.jsonl`.
- Nhiều kết quả retry của cùng URL: dùng record thành công cuối cùng.
- Chỉ giữ record `success=True`, khử trùng URL và ghi atomically vào `/kaggle/working/retry-antibot-output/dataset.jsonl`.

Khi `DRY_RUN=False`, merge được bật mặc định. Có thể đặt `APPLY_DATASET_MERGE=False` để chỉ preview.

In [ ]:
import os
APPLY_DATASET_MERGE = not DRY_RUN

def read_jsonl_records(path):
    if not path.exists():
        return []
    records = []
    with path.open(encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON lỗi tại {path.name}:{line_no}") from exc
    return records

def prepare_dataset_record(retry_record, source_record=None, existing_record=None):
    # Thứ tự overlay: metadata attempt -> record dataset cũ -> nội dung retry mới.
    merged = dict(source_record or {})
    merged.update(existing_record or {})
    merged.update(retry_record)

    markdown = merged.get("markdown") or ""
    merged.update({
        "success": True,
        "error_type": None,
        "error": None,
        "raw_markdown": markdown,
        "fit_markdown": merged.get("fit_markdown") or "",
        "markdown_source": "retry_raw",
        "markdown_chars": len(markdown),
        "raw_markdown_chars": len(markdown),
        "fit_markdown_chars": len(merged.get("fit_markdown") or ""),
        "final_status_code": merged.get("final_status_code") or merged.get("status_code"),
    })
    merged.pop("response_preview", None)
    return merged

def build_merged_dataset(dataset_path=SOURCE_DATASET_PATH, retry_path=SUCCESS_PATH):
    original = [
        row for row in read_jsonl_records(dataset_path)
        if row.get("success") is True and row.get("url")
    ]
    recovered_rows = [r for r in read_jsonl_records(retry_path) if r.get("success") is True and r.get("url")]

    # Dict giữ insertion order; gán lại giúp record cuối cùng thắng.
    latest_retry_by_url = {}
    for row in recovered_rows:
        latest_retry_by_url[row["url"]] = row

    source_by_url = {}
    for row in read_jsonl_records(ATTEMPTS_PATH):
        if row.get("url"):
            source_by_url[row["url"]] = row

    merged_rows = []
    seen_urls = set()
    replaced = 0
    for old_row in original:
        url = old_row.get("url")
        if url in seen_urls:
            continue
        seen_urls.add(url)
        retry_row = latest_retry_by_url.get(url)
        if retry_row:
            merged_rows.append(prepare_dataset_record(retry_row, source_by_url.get(url), old_row))
            replaced += 1
        else:
            merged_rows.append(old_row)

    appended = 0
    for url, retry_row in latest_retry_by_url.items():
        if url in seen_urls:
            continue
        merged_rows.append(prepare_dataset_record(retry_row, source_by_url.get(url)))
        seen_urls.add(url)
        appended += 1

    stats = {
        "source_success_rows": len(original),
        "source_unique_urls": len({row["url"] for row in original}),
        "unique_retry_success_urls": len(latest_retry_by_url),
        "replaced_rows": replaced,
        "appended_rows": appended,
        "final_rows": len(merged_rows),
    }
    return merged_rows, stats

def atomic_write_dataset(records, dataset_path=DATASET_PATH):
    temp_path = dataset_path.with_name(f".{dataset_path.name}.tmp")

    try:
        with temp_path.open("w", encoding="utf-8") as f:
            for record in records:
                f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
            f.flush()
            os.fsync(f.fileno())
        os.replace(temp_path, dataset_path)
    finally:
        if temp_path.exists():
            temp_path.unlink()
    return dataset_path

merged_dataset, merge_stats = build_merged_dataset()
dataset_urls = [row["url"] for row in merged_dataset]
assert all(row.get("success") is True for row in merged_dataset), "Dataset còn record lỗi"
assert len(dataset_urls) == len(set(dataset_urls)), "Dataset còn URL trùng"
assert merge_stats["final_rows"] == merge_stats["source_unique_urls"] + merge_stats["appended_rows"], "Sai số lượng sau merge"
print(merge_stats)

if APPLY_DATASET_MERGE:
    atomic_write_dataset(merged_dataset)
    print(f"Đã cập nhật: {DATASET_PATH}")
else:
    print("Preview only: dataset.jsonl chưa được thay đổi.")

target_counts = attempts.loc[attempts[domain_col].astype(str).isin(zero_domains)].groupby(domain_col)["url"].nunique()
success_counts = retry_success.groupby("sampled_domain")["url"].nunique() if not retry_success.empty else pd.Series(dtype=int)
error_counts = retry_errors.groupby("sampled_domain")["url"].nunique() if not retry_errors.empty else pd.Series(dtype=int)
domain_report = pd.DataFrame({"target_urls": target_counts, "retry_success": success_counts, "still_failed": error_counts}).fillna(0).astype(int)
domain_report.index.name = domain_col
domain_report["pending"] = (domain_report["target_urls"] - domain_report["retry_success"] - domain_report["still_failed"]).clip(lower=0)
domain_report["added_to_dataset"] = domain_report["retry_success"]
display(domain_report.reset_index().sort_values(domain_col))


## 8. Đóng gói kết quả Kaggle

Cell này tạo một file ZIP trong `/kaggle/working` chứa dataset cuối và các log. Sau khi notebook chạy xong, file cũng xuất hiện trong tab **Output** của Kaggle.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile
from IPython.display import FileLink

ARTIFACT_PATH = (
    Path("/kaggle/working/retry-antibot-results.zip")
    if IS_KAGGLE else OUTPUT_DIR / "retry-antibot-results.zip"
)
artifact_files = [DATASET_PATH, SUCCESS_PATH, ERROR_PATH, EVENT_LOG_PATH]

with ZipFile(ARTIFACT_PATH, "w", compression=ZIP_DEFLATED) as archive:
    for path in artifact_files:
        if path.exists():
            archive.write(path, arcname=path.name)

print(f"Artifact: {ARTIFACT_PATH} ({ARTIFACT_PATH.stat().st_size / 1024 / 1024:.2f} MB)")
display(FileLink(str(ARTIFACT_PATH)))


## Khi vẫn bị chặn

1. Chạy thử `HEADLESS=False` trên máy local để nhìn chính xác trang trả về.
2. Nếu trang yêu cầu CAPTCHA/login, hãy dừng automation. Có thể tạo profile hợp lệ thủ công bằng `crwl profiles`, rồi đặt `PROFILE_DIR` về profile đó nếu bạn có quyền truy cập.
3. Nếu cả 3–5 URL pilot của một domain đều bị chặn, không tăng concurrency. Liên hệ chủ website, dùng API/feed/sitemap công khai hoặc xin quyền crawl.
4. `403` do `robots.txt` không phải lỗi cần bypass. Giữ nguyên trạng thái thất bại.
5. Chỉ cân nhắc proxy do tổ chức sở hữu/được phép sử dụng; notebook này cố ý không cấu hình proxy rotation.